In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns

from network_original_lpn import LPN
from network_ne_by_norm import NE_LPN_By_Norm
from network_ne_by_design import NE_LPN_By_Design
from utils import prox

sns.set()

MODEL_DIR = "experiments/models/"
PLOT_DIR = "experiments/plots/"
os.makedirs(PLOT_DIR, exist_ok=True)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
class NormalSampler:
    def __init__(self, mean=0.0, std=1.0):
        self.mean = mean
        self.std = std

    def __call__(self, n):
        return torch.randn(n) * self.std + self.mean

def add_noise(x, sigma=0.1):
    return x + torch.randn_like(x) * sigma

In [ ]:
def test_model(model, model_name, sampler, sigma_noise=1.0):
    model.eval()
    model = model.to(device)
    
    with torch.no_grad():
        test_clean = sampler(10000).unsqueeze(1).to(device)
        test_noisy = add_noise(test_clean, sigma_noise).to(device)
        test_denoised = model(test_noisy)

        mse = torch.nn.functional.mse_loss(test_denoised, test_clean).item()

        plt.figure(figsize=(3, 3))
        plt.scatter(test_noisy.cpu(), test_clean.cpu(), label="clean target", s=4, alpha=0.5)
        plt.scatter(test_noisy.cpu(), test_denoised.cpu(), label="denoised", s=4, alpha=0.5)
        plt.plot([-4, 4], [-4, 4], 'k--', lw=1)
        plt.legend()
        plt.title(f"{model_name} | MSE: {mse:.4f}")
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(PLOT_DIR, f"{model_name}_test_scatter.png"))
        plt.show()

        return mse

In [ ]:
sampler = NormalSampler()
sigma_noise = 1.0

dim = 1
hidden = 50
layers = 4
beta = 10

In [ ]:
model_defs = {
    "Original_LPN": LPN,
    "NE_By_Norm": NE_LPN_By_Norm,
    "NE_By_Design": NE_LPN_By_Design
}

In [ ]:
results = {}
for name, cls in model_defs.items():
    model = cls(in_dim=dim, hidden=hidden, layers=layers, beta=beta)
    model.load_state_dict(torch.load(os.path.join(MODEL_DIR, f"{name}.pth"), map_location=device))
    mse = test_model(model, name, sampler, sigma_noise=sigma_noise)
    results[name] = mse

print("\nFinal Test MSEs:")
for name, mse in results.items():
    print(f"{name}: {mse:.6f}")